# CVPR 2026 HOW Workshop — Toward a Shared Stack for Vision Interpretability

Companion notebook for the talk at the **CVPR 2026 HOW Workshop**.
Source + standalone CLIs for every section live at
[github.com/JadenFiotto-Kaufman/CVPR2026-HOW](https://github.com/JadenFiotto-Kaufman/CVPR2026-HOW).

> **Runtime**: this notebook is sized for a free Colab **T4** instance
> (Runtime → Change runtime type → T4 GPU). Section 1 runs a small SD
> 1.4 model locally; Sections 2 and 3 ship traces to a remote NDIF, so
> the kernel only needs CPU + a few hundred MB of GPU memory.

**Contents**

- **Section 1 — Attention ablation (a tour of `nnsight`)**
  What does each cross-attention layer in Stable Diffusion actually
  contribute? Zero one (or several) and watch the image change. The
  walkthrough doubles as the `nnsight` API intro — `.trace()`,
  `.input` / `.output`, `.save()`, `tracer.iter[:]`.
- **Section 2 — Concept attention (using NDIF)**
  Interpretable per-concept heatmaps for FLUX.2. The whole 4B-parameter
  pipeline runs on NDIF — your Colab only does tokenisation and a
  handful of tiny tensor ops.
- **Section 3 — VLM logit lens (using Workbench)**
  Read per-layer next-token predictions out of LLaVA, including over the
  576 image-patch positions, and render an interactive heatmap.

## Install

`nnsight` from the `dev` branch (this notebook uses APIs newer than the
last PyPI release). `diffusers` / `transformers` / `accelerate` are
needed for the model wrappers and pipelines. The last line pulls the
notebook's companion `util.py` (small visualization helpers) — Colab
only loads the `.ipynb` itself when you open from GitHub, so sibling
files have to be fetched explicitly.

In [ ]:
from IPython.display import clear_output, display

#!pip install -q git+https://github.com/ndif-team/nnsight.git@dev
#!pip install -q --upgrade diffusers transformers accelerate
#!wget -q https://raw.githubusercontent.com/JadenFiotto-Kaufman/CVPR2026-HOW/master/colab/util.py

# clear_output()

import util

# Section 1 — Attention ablation (a tour of `nnsight`)

Stable Diffusion 1.4's UNet has **16 cross-attention layers**
(every `transformer_block.attn2`) sprinkled across its down-blocks,
mid-block, and up-blocks. Cross-attention is the only place where the
text prompt directly steers the image stream — so a natural
interpretability question is: *what does each individual cross-attention
layer contribute to the final image?*

The recipe with `nnsight`: open a `model.generate(...)` trace, iterate
over the denoising steps with `tracer.iter[:]`, and inside each step
zero the output of whichever cross-attention layers you want to ablate.
The image stream's forward pass everywhere else is untouched.

This is also the first time `nnsight` shows up in the notebook, so the
walkthrough below introduces every part of the core API as it gets used.

The library reduces to one core pattern:

1. Wrap any PyTorch model with `Envoy` (or, for tighter HuggingFace
   integration, use one of the model-family wrappers — `LanguageModel`,
   `VisionLanguageModel`, `DiffusionModel`, ...).
2. Open a `with model.trace(input):` (or `model.generate(input):`) block.
3. Inside the block, write normal Python that **references** the
   activations you want to read or change — accessing `module.output`,
   `module.input`, doing arithmetic on them, slicing them, replacing
   them. Use `.save()` on any variable to access it after the with block.
4. Exit the block. The model executes, your interventions are applied,
   and saved activations are accessible as real torch tensors.

Key surface:

| What | Returns |
|---|---|
| `module.output` | The output of a module's forward pass. |
| `module.input`  | The first positional arg the module receives. |
| `module.inputs` | A `(args, kwargs)` pair — what the module is *actually* called with. |
| `.save()`       | Keeps a value alive past the trace exit. Without it, values get filtered out on cleanup. |

Original work: [github.com/JadenFiotto-Kaufman/thesis](https://github.com/JadenFiotto-Kaufman/thesis).

![Diffusion](Diffusion.webp)

## Load the model

`DiffusionModel(...)` wraps a HuggingFace `DiffusionPipeline` so every
sub-module (the UNet, its attention blocks, the text encoder, ...) is
accessible as an `Envoy` — the object you reference inside a trace to
read activations or install interventions. `dispatch=True` materializes
the weights on the chosen device right now; `dispatch=False` would keep
them on `meta` for remote execution against NDIF (see Section 2).

from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="CompVis/stable-diffusion-v1-4",
    local_dir=r"D:\work\python\CVPR-2026_HOW_Workshop\stable-diffusion-v1-4",
    local_dir_use_symlinks=False,
    allow_patterns=["vae/*"],
)

In [ ]:
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from nnsight import DiffusionModel

local_path = Path("models/stable-diffusion-v1-4")
model_ref = str(local_path)

sd = DiffusionModel(
    model_ref,
    torch_dtype=torch.float16,
    use_safetensors=False,
    dispatch=True,
    device_map="cuda",
)



## Baseline (no ablation)

Generate the reference image first so we have something to diff
against. With no interventions in play, `sd.generate(...)` behaves
exactly like the underlying `DiffusionPipeline.__call__` — call it
directly and read `.images[0]` off the result. We'll reach for the
`with sd.generate(...) as tracer:` form below, once we actually have
something to do *inside* the trace.

In [ ]:
PROMPT = "The dog is sitting on a hat"
PROMPT = "A dog wearing a hat"
PROMPT = "Dog wearing a hat, in the style of Van Gogh"
PROMPT = "Starry Night"

SEED = 43
NUM_INFERENCE_STEPS = 50
baseline_image = sd.generate(
    PROMPT, num_inference_steps=NUM_INFERENCE_STEPS, seed=SEED,
).images[0]
print(baseline_image.size)
display(baseline_image)

## List the cross-attention layers

`named_modules` returns every sub-Envoy in the UNet; we keep the ones
whose path ends in `.attn2` (cross-attention), sorted into a stable
forward-pass-friendly order (down → mid → up).

In [ ]:
cross_attentions = sorted(
    [
        (name, envoy)
        for name, envoy in sd.unet.named_modules()
        if name.endswith(".attn2")
    ],
    key=lambda x: x[0], # sort by the name (Down, Min, Up)
)

cross_attention_envoys = [envoy for _, envoy in cross_attentions]

for i, (name, _) in enumerate(cross_attentions):
    print(f"  [{i:2d}] {name}")

## Ablate the chosen layers

Edit `LAYERS_TO_ABLATE` below — each index corresponds to a row in the
list above. Defaults to `[5]`, which on the `"Starry Night"` prompt
strips the painterly Van-Gogh association out of the generation and
leaves a generic star-filled night sky behind. No other single cross-
attention layer has this effect — suggesting layer 5 is where the
specific *painting* (vs. the literal concept "stars at night")
gets bound.

Three pieces of `nnsight` come together here:

* `tracer.iter[:]` — diffusion runs the UNet once per denoising step,
  so we need an intervention that fires *every step*, not just on the
  first forward pass.
* `envoy.to_out[0].input` — `.input` is the tensor passed *into* a
  module's forward. `to_out[0]` is the cross-attention's output linear
  projection, so its input is the post-SDPA, pre-projection activation.
* `... [:] = 0` — slice-assignment on a captured tensor is an
  intervention: downstream code (the projection and everything after)
  sees zeros. The model still runs and attention is still computed; its
  result just doesn't get added back into the image stream.

In [ ]:
import util
LAYERS_TO_ABLATE = [5]  # Edit me — e.g. [0, 7, 15] to ablate several at once.
LAYERS_TO_ABLATE = [5, 8]  # Edit me — e.g. [0, 7, 15] to ablate several at once.
# LAYERS_TO_ABLATE = [0,1,2,3,4]  # Edit me — e.g. [0, 7, 15] to ablate several at once.

#LAYERS_TO_ABLATE = list(range(7,16))
with sd.generate(PROMPT, num_inference_steps=NUM_INFERENCE_STEPS, seed=SEED) as tracer:
    for _step in tracer.iter[:]:
        # Sort to access in forward order — nnsight's one-shot hooks
        # are order-sensitive within a single forward pass.
        for idx in sorted(LAYERS_TO_ABLATE):
            cross_attention_envoys[idx].to_out[0].input[:] = 0
    ablated = tracer.result.save()

ablated_image = ablated.images[0]

util.show_ablation_comparison(baseline_image, ablated_image, LAYERS_TO_ABLATE)

## Sweep — ablate every cross-attention layer in turn

To see whether layer 5's effect is special or just one point on a
spectrum, repeat the experiment for **every** cross-attention block —
one ablated layer per generation, same prompt and seed — and lay the
results out as a grid. Each tile is the same `"Starry Night"` prompt
with exactly that one cross-attention killed; the baseline tile sits
in the top-left for reference.

Expect to spend ~1–2 s per tile on a Colab T4, so ~30 s for all 16.

In [ ]:
per_layer_images = []
for layer_idx in range(len(cross_attention_envoys)):
    with sd.generate(
        PROMPT, num_inference_steps=NUM_INFERENCE_STEPS, seed=SEED,
    ) as tracer:
        for _step in tracer.iter[:]:
            cross_attention_envoys[layer_idx].to_out[0].input[:] = 0
        out = tracer.result.save()
    per_layer_images.append(out.images[0])

util.show_per_layer_ablation_grid(baseline_image, per_layer_images)